# Qwen2.5-VL Step-DPO Fine-Tuning (2xT4 on Kaggle)

This notebook trains a QLoRA adapter on the extracted Step-DPO pairs to align the model's reasoning format.

In [ ]:
!pip install -qU transformers accelerate peft bitsandbytes trl datasets flash-attn
!pip install -q qwen-vl-utils

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

# 1. Load Dataset
data_path = "experiments/001_500_reasoning/data/step_dpo_pairs.jsonl"
with open(data_path, 'r') as f:
    raw_data = [json.loads(line) for line in f]
    
hf_data = {"prompt": [], "chosen": [], "rejected": []}
for item in raw_data:
    # DPO requires prompt to be formatted as the chat context
    prompt_content = [
        {"type": "image", "image": item["image_path"]},
        {"type": "text", "text": "Analyze this chart. Provide step-by-step reasoning and a final answer.\n" + item.get("question", "")}
    ]
    prompt_msg = [{"role": "user", "content": prompt_content}]
    
    # Append the assistant's prefix (correct steps up to divergence)
    if item["prefix"].strip():
        prompt_msg.append({"role": "assistant", "content": item["prefix"]})
        
    hf_data["prompt"].append(prompt_msg)
    
    # Qwen processor requires conversational format even for chosen/rejected in trl DPO
    # For DPO, we just need the string completion because TRL's data collator handles it.
    hf_data["chosen"].append([{"role": "assistant", "content": item["chosen"] + "\n"}])
    hf_data["rejected"].append([{"role": "assistant", "content": item["rejected"] + "\n"}])

dataset = Dataset.from_dict(hf_data)
print(f"Loaded {len(dataset)} pairs.")

In [ ]:
# 2. Load Model and Processor
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# device_map="auto" automatically distributes the model across both T4 GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
# Prevent DPO from padding infinitely
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

In [ ]:
# 3. Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

In [ ]:
# 4. Define DPO Trainer
training_args = DPOConfig(
    output_dir="./dpo_qwen_vl",
    beta=0.1, # KL penalty
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=2,
    logging_steps=1, # Verbose logging as requested
    save_steps=10,   # Frequent checkpointing
    max_prompt_length=1024,
    max_length=1536,
    remove_unused_columns=False,
    report_to="none",
)

def formatting_func(example):
    # Formatting function for DPO in trl with chat templates
    prompt = processor.apply_chat_template(example["prompt"], tokenize=False, add_generation_prompt=True)
    # Note: TRL expects raw strings for chosen/rejected if chat template is already applied to prompt
    chosen = example["chosen"][0]["content"]
    rejected = example["rejected"][0]["content"]
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

formatted_dataset = dataset.map(formatting_func)

trainer = DPOTrainer(
    model,
    ref_model=None, # PEFT handles reference automatically
    args=training_args,
    train_dataset=formatted_dataset,
    tokenizer=processor.tokenizer,
    peft_config=peft_config,
)

In [ ]:
# 5. Train and Save
trainer.train()
trainer.save_model("qwen_vl_step_dpo_adapter")
print("Training complete and adapter saved.")